# 04 — TMDB Keyword Enrichment: Movie Usage

Adds `movie_count` to the keyword enrichment table produced by `00_tmdb_keyword_download`.

**Inputs**
- `output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv` — from notebook `00`
- `data/tmdb_movie_keyword_pairs.csv` — from notebook `03`

**Output**
- `output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv` — updated in-place with `movie_count` column

| Column | Description |
|---|---|
| `tmdb_keyword_id` | TMDb canonical keyword ID |
| `name` | Keyword text |
| `keyword_type` | Taxonomy label |
| `lexical_class` | Vocabulary class |
| `min_zipf_freq` | Rarity proxy (min token Zipf frequency) |
| `is_narrative` | True if keyword carries narrative signal |
| `movie_count` | Distinct movies this keyword appears in |


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

KEYWORDS_FILE = Path("output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv")
PAIRS_FILE    = Path("data/tmdb_movie_keyword_pairs.csv")


## Load

In [2]:
keywords = pd.read_csv(KEYWORDS_FILE)
# Drop stale movie_count if re-running
if "movie_count" in keywords.columns:
    keywords = keywords.drop(columns=["movie_count"])

pairs = pd.read_csv(PAIRS_FILE)

print(f"Keywords : {len(keywords):,}")
print(f"Pairs    : {len(pairs):,}")


Keywords : 84,842
Pairs    : 928,781


## Join movie_count

In [3]:
movie_freq = (
    pairs.groupby("tmdb_keyword_id")["tmdb_movie_id"]
    .nunique()
    .rename("movie_count")
    .reset_index()
)

keywords = keywords.merge(movie_freq, on="tmdb_keyword_id", how="left")
keywords["movie_count"] = keywords["movie_count"].fillna(0).astype(int)

used    = (keywords["movie_count"] > 0).sum()
unused  = (keywords["movie_count"] == 0).sum()
top     = keywords.loc[keywords["movie_count"].idxmax()]
print(f"Keywords used in 1+ movies : {used:,}")
print(f"Keywords never used        : {unused:,}")
print(f"Most used: '{top['name']}' — {top['movie_count']:,} movies")
keywords.head(5)


Keywords used in 1+ movies : 59,070
Keywords never used        : 25,772
Most used: 'short film' — 28,371 movies


,tmdb_keyword_id,name,keyword_type,lexical_class,min_zipf_freq,is_narrative,movie_count
0,30,individual,theme_or_plot,common,5.00,True,43
1,65,holiday,theme_or_plot,common,4.66,True,1067
2,74,germany,location,common,4.90,True,1033
3,75,gunslinger,object_motif,non_standard,2.58,True,78
4,83,saving the world,plot_event,common,4.59,True,81


## Distribution

In [4]:
buckets = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, 5000, float('inf')]
labels  = ['1','2','3-5','6-10','11-25','26-50','51-100','101-250','251-500','501-1k','1k-5k','5k+']
kw_used = keywords[keywords["movie_count"] > 0].copy()
kw_used["bucket"] = pd.cut(kw_used["movie_count"], bins=buckets, labels=labels)

dist = kw_used["bucket"].value_counts().reindex(labels).reset_index()
dist.columns = ["movies_per_keyword", "keyword_count"]

fig = px.bar(
    dist, x="movies_per_keyword", y="keyword_count", text_auto=True,
    title="Keyword usage distribution (movies per keyword)",
    labels={"movies_per_keyword": "Movies using keyword", "keyword_count": "Number of keywords"},
)
fig.show()


In [5]:
print("Top 20 most used keywords:")
display(
    keywords.nlargest(20, "movie_count")[["name", "keyword_type", "is_narrative", "movie_count"]]
    .reset_index(drop=True)
)


Top 20 most used keywords:


,name,keyword_type,is_narrative,movie_count
0,short film,production_meta,False,28371
1,woman director,production_meta,False,15458
2,based on novel or book,production_meta,False,6485
3,lgbt,theme_or_plot,True,5109
4,concert,production_meta,False,5061
5,murder,theme_or_plot,True,4829
6,biography,production_meta,False,4029
7,silent film,production_meta,False,3762
8,sports,theme_or_plot,True,3614
9,stand-up comedy,genre_as_keyword,False,3603


In [6]:
fig2 = px.histogram(
    kw_used, x="movie_count", color="is_narrative",
    nbins=50, log_x=True, log_y=True, barmode="overlay", opacity=0.7,
    title="Movie count per keyword (log-log) — narrative vs non-narrative",
    labels={"movie_count": "Movies per keyword (log)"},
)
fig2.show()


## Save

In [7]:
keywords.to_csv(KEYWORDS_FILE, index=False)
print(f"Saved {len(keywords):,} rows → {KEYWORDS_FILE}")
print(f"Columns: {list(keywords.columns)}")


Saved 84,842 rows → output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv
Columns: ['tmdb_keyword_id', 'name', 'keyword_type', 'lexical_class', 'min_zipf_freq', 'is_narrative', 'movie_count']


## Upload to Kaggle

In [8]:
import os

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub
    from datetime import date
    user = kagglehub.whoami(verbose=False)["username"]
    kagglehub.dataset_upload(
        handle=f"{user}/tmdb-keyword-enriched",
        local_dataset_dir=str(KEYWORDS_FILE.parent),
        version_notes=f"Add movie_count column — {date.today()}",
        ignore_patterns=["*.json"],
    )
    print(f"→ kaggle.com/datasets/{user}/tmdb-keyword-enriched")


Skipping upload: set KAGGLE_TOKEN env var to upload.
